# Install dependencies

In [17]:
import sys
import subprocess
import importlib.util

packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "PIL": "pillow",
    "tqdm": "tqdm",
    "tabulate": "tabulate"
}

missing = [pip_name for module_name, pip_name in packages.items() if importlib.util.find_spec(module_name) is None]

if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

print("Dependency check completed.")


Dependency check completed.


# Imports

In [3]:
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from datetime import datetime
from collections import defaultdict
from tqdm.auto import tqdm
import pandas as pd
import numpy as np
import json
import os
import sys
import shutil
import platform
import hashlib
import math
import warnings

Image.MAX_IMAGE_PIXELS = 300_000_000
warnings.simplefilter("error", Image.DecompressionBombWarning)

print("Imports completed.")

Imports completed.


# Configuration

In [4]:
PROJECT_DIR = Path("./anime_character_pipeline").expanduser().resolve()
PREVIOUS_RUN_DIR = None

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = PROJECT_DIR / "runs" / f"05_folder_materialization_{RUN_ID}"

REPORTS_DIR = RUN_DIR / "reports"
TABLES_DIR = RUN_DIR / "tables"
LOGS_DIR = RUN_DIR / "logs"

FINAL_OUTPUT_DIR = PROJECT_DIR / "organized_output" / f"anime_organized_{RUN_ID}"

MATERIALIZATION_MODE = "hardlink"
ALLOW_HARDLINK_FALLBACK_TO_COPY = True
ON_EXISTING = "error"

SOURCE_COLUMN = "source_path"

CREATE_CLUSTER_METADATA_FILES = True
CREATE_FOLDER_MANIFEST_FILES = True
CREATE_OUTPUT_README = True
CREATE_OUTPUT_CONTACT_SHEET_INDEX = True

COPY_CLUSTER_CONTACT_SHEETS_TO_OUTPUT = True

CONTACT_SHEET_THUMB_SIZE = 170
CONTACT_SHEET_COLUMNS = 6
CONTACT_SHEET_MAX_IMAGES_PER_FOLDER = 24
MAX_FOLDER_CONTACT_SHEETS = 300

VALID_MATERIALIZATION_MODES = {"hardlink", "copy", "symlink"}
VALID_ON_EXISTING = {"error", "skip", "overwrite"}

if MATERIALIZATION_MODE not in VALID_MATERIALIZATION_MODES:
    raise ValueError(f"Invalid MATERIALIZATION_MODE: {MATERIALIZATION_MODE}")

if ON_EXISTING not in VALID_ON_EXISTING:
    raise ValueError(f"Invalid ON_EXISTING: {ON_EXISTING}")

for directory in [RUN_DIR, REPORTS_DIR, TABLES_DIR, LOGS_DIR, FINAL_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Project directory: {PROJECT_DIR}")
print(f"Run directory: {RUN_DIR}")
print(f"Final output directory: {FINAL_OUTPUT_DIR}")
print(f"Materialization mode: {MATERIALIZATION_MODE}")

Project directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline
Run directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materialization_20260523_220036
Final output directory: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/organized_output/anime_organized_20260523_220036
Materialization mode: hardlink


# Locate previous clustering run and load manifests

In [5]:
def find_latest_clustering_run(project_dir):
    runs_dir = Path(project_dir) / "runs"
    candidates = sorted(
        [p for p in runs_dir.glob("04_hdbscan_clustering_*") if p.is_dir()],
        key=lambda p: p.name,
    )

    valid_candidates = []
    for candidate in candidates:
        assignment_path = candidate / "tables" / "folder_assignment_manifest.csv"
        cluster_summary_path = candidate / "tables" / "cluster_summary.csv"
        if assignment_path.exists() and cluster_summary_path.exists():
            valid_candidates.append(candidate)

    if not valid_candidates:
        raise FileNotFoundError(
            f"No valid 04_hdbscan_clustering_* run was found in {runs_dir}. "
            "Run notebook 04 before this notebook."
        )

    return valid_candidates[-1]


PREVIOUS_RUN_DIR = globals().get("PREVIOUS_RUN_DIR", None)

if PREVIOUS_RUN_DIR is None:
    PREVIOUS_RUN_DIR = find_latest_clustering_run(PROJECT_DIR)
else:
    PREVIOUS_RUN_DIR = Path(PREVIOUS_RUN_DIR).expanduser().resolve()

previous_tables_dir = PREVIOUS_RUN_DIR / "tables"

folder_assignment_path = previous_tables_dir / "folder_assignment_manifest.csv"
cluster_summary_path = previous_tables_dir / "cluster_summary.csv"
cluster_contact_sheets_path = previous_tables_dir / "cluster_contact_sheets.csv"
special_contact_sheets_path = previous_tables_dir / "special_contact_sheets.csv"

if not folder_assignment_path.exists():
    raise FileNotFoundError(f"Required folder assignment manifest not found: {folder_assignment_path}")

if not cluster_summary_path.exists():
    raise FileNotFoundError(f"Required cluster summary not found: {cluster_summary_path}")

folder_assignment_df = pd.read_csv(folder_assignment_path)
cluster_summary_df = pd.read_csv(cluster_summary_path)

if cluster_contact_sheets_path.exists():
    cluster_contact_sheets_df = pd.read_csv(cluster_contact_sheets_path)
else:
    cluster_contact_sheets_df = pd.DataFrame()

if special_contact_sheets_path.exists():
    special_contact_sheets_df = pd.read_csv(special_contact_sheets_path)
else:
    special_contact_sheets_df = pd.DataFrame()

print(f"Previous clustering run: {PREVIOUS_RUN_DIR}")
print(f"Folder assignment rows loaded: {len(folder_assignment_df)}")
print(f"Cluster summary rows loaded: {len(cluster_summary_df)}")
print(f"Cluster contact sheet rows loaded: {len(cluster_contact_sheets_df)}")
print(f"Special contact sheet rows loaded: {len(special_contact_sheets_df)}")

Previous clustering run: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/04_hdbscan_clustering_20260523_214927
Folder assignment rows loaded: 15705
Cluster summary rows loaded: 189
Cluster contact sheet rows loaded: 188
Special contact sheet rows loaded: 4


# Helper functions

In [6]:
def now_iso():
    return datetime.now().astimezone().isoformat(timespec="seconds")


def normalize_path_string(value):
    return Path(str(value)).expanduser().resolve().as_posix()


def stable_short_hash(value, length=12):
    value = str(value).encode("utf-8", errors="replace")
    return hashlib.sha256(value).hexdigest()[:length]


def safe_bool(value):
    if isinstance(value, bool):
        return value
    if pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes", "y"}


def sanitize_filename_component(value, fallback="item", max_length=140):
    value = str(value).strip()

    if not value:
        value = fallback

    forbidden = '<>:"/\\|?*'
    cleaned = "".join("_" if ch in forbidden else ch for ch in value)
    cleaned = "".join(ch if ch.isprintable() else "_" for ch in cleaned)
    cleaned = cleaned.strip().strip(".")
    cleaned = "_".join(cleaned.split())

    if not cleaned:
        cleaned = fallback

    return cleaned[:max_length]


def sanitize_folder_name(value, fallback="_needs_review", max_length=120):
    return sanitize_filename_component(value, fallback=fallback, max_length=max_length)


def make_destination_filename(row, source_path):
    source_path = Path(source_path)
    suffix = source_path.suffix
    stem = source_path.stem

    embedding_row = row.get("embedding_row", None)

    try:
        embedding_row = int(embedding_row)
        prefix = f"{embedding_row:07d}"
    except Exception:
        prefix = stable_short_hash(source_path.as_posix(), length=10)

    safe_stem = sanitize_filename_component(stem, fallback="image", max_length=110)
    safe_suffix = suffix if suffix else ""

    return f"{prefix}__{safe_stem}{safe_suffix}"


def ensure_unique_destination_path(destination_path, used_paths):
    destination_path = Path(destination_path)
    destination_key = destination_path.as_posix()

    if destination_key not in used_paths and not destination_path.exists():
        used_paths.add(destination_key)
        return destination_path

    parent = destination_path.parent
    stem = destination_path.stem
    suffix = destination_path.suffix

    counter = 1

    while True:
        candidate = parent / f"{stem}__dup{counter:03d}{suffix}"
        candidate_key = candidate.as_posix()

        if candidate_key not in used_paths and not candidate.exists():
            used_paths.add(candidate_key)
            return candidate

        counter += 1


def remove_existing_file(path):
    path = Path(path)

    if path.is_symlink() or path.is_file():
        path.unlink()
    elif path.exists():
        raise IsADirectoryError(f"Destination exists and is not a regular file or symlink: {path}")


def materialize_one_file(source_path, destination_path, mode, on_existing, allow_hardlink_fallback_to_copy):
    source_path = Path(source_path)
    destination_path = Path(destination_path)

    result = {
        "status": "unknown",
        "operation_used": None,
        "error_type": None,
        "error_message": None,
    }

    try:
        if not source_path.exists():
            raise FileNotFoundError(f"Source file does not exist: {source_path}")

        if not source_path.is_file():
            raise ValueError(f"Source path is not a regular file: {source_path}")

        destination_path.parent.mkdir(parents=True, exist_ok=True)

        if destination_path.exists() or destination_path.is_symlink():
            if on_existing == "error":
                raise FileExistsError(f"Destination already exists: {destination_path}")
            if on_existing == "skip":
                result["status"] = "skipped_existing"
                result["operation_used"] = "skip"
                return result
            if on_existing == "overwrite":
                remove_existing_file(destination_path)

        if mode == "copy":
            shutil.copy2(source_path, destination_path)
            result["status"] = "ok"
            result["operation_used"] = "copy"
            return result

        if mode == "symlink":
            destination_path.symlink_to(source_path)
            result["status"] = "ok"
            result["operation_used"] = "symlink"
            return result

        if mode == "hardlink":
            try:
                os.link(source_path, destination_path)
                result["status"] = "ok"
                result["operation_used"] = "hardlink"
                return result
            except OSError as hardlink_exc:
                if not allow_hardlink_fallback_to_copy:
                    raise

                shutil.copy2(source_path, destination_path)
                result["status"] = "ok"
                result["operation_used"] = "copy_fallback_from_hardlink"
                result["error_type"] = type(hardlink_exc).__name__
                result["error_message"] = str(hardlink_exc)
                return result

        raise ValueError(f"Unsupported materialization mode: {mode}")

    except Exception as exc:
        result["status"] = "error"
        result["operation_used"] = mode
        result["error_type"] = type(exc).__name__
        result["error_message"] = str(exc)
        return result


def create_contact_sheet(image_paths, output_path, thumb_size=170, columns=6, title=None):
    valid_paths = [Path(p) for p in image_paths if pd.notna(p) and Path(p).exists()]

    if not valid_paths:
        return False

    rows = int(math.ceil(len(valid_paths) / columns))
    title_height = 34 if title else 0

    sheet_width = columns * thumb_size
    sheet_height = rows * thumb_size + title_height

    sheet = Image.new("RGB", (sheet_width, sheet_height), "white")
    draw = ImageDraw.Draw(sheet)

    if title:
        draw.rectangle((0, 0, sheet_width, title_height), fill=(245, 245, 245))
        draw.text((10, 9), title[:180], fill=(20, 20, 20))

    for idx, path in enumerate(valid_paths):
        try:
            with Image.open(path) as img:
                img = ImageOps.exif_transpose(img.convert("RGB"))
                img.thumbnail((thumb_size, thumb_size), Image.Resampling.LANCZOS)

                x = (idx % columns) * thumb_size + (thumb_size - img.width) // 2
                y = title_height + (idx // columns) * thumb_size + (thumb_size - img.height) // 2

                sheet.paste(img, (x, y))
        except Exception:
            continue

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(output_path, format="JPEG", quality=92, optimize=True)

    return True


def dataframe_to_json_records(df, max_rows=None):
    if max_rows is not None:
        df = df.head(max_rows).copy()

    records = df.replace({np.nan: None}).to_dict(orient="records")
    return records


print("Helper functions loaded.")

Helper functions loaded.


# Validate required columns and prepare assignment table


In [7]:
required_columns = {
    "embedding_row",
    "relative_path",
    "source_path",
    "crop_path",
    "cluster_label",
    "cluster_probability",
    "outlier_score",
    "requires_manual_review",
    "assignment_category",
    "proposed_folder",
}

missing_columns = sorted(required_columns.difference(set(folder_assignment_df.columns)))

if missing_columns:
    raise ValueError(f"The folder assignment manifest is missing required columns: {missing_columns}")

if SOURCE_COLUMN not in folder_assignment_df.columns:
    raise ValueError(f"SOURCE_COLUMN does not exist in folder_assignment_df: {SOURCE_COLUMN}")

assignment_df = folder_assignment_df.copy()

assignment_df[SOURCE_COLUMN] = assignment_df[SOURCE_COLUMN].map(normalize_path_string)
assignment_df["proposed_folder"] = assignment_df["proposed_folder"].map(
    lambda value: sanitize_folder_name(value, fallback="_needs_review")
)
assignment_df["requires_manual_review"] = assignment_df["requires_manual_review"].map(safe_bool)

assignment_df["source_exists"] = assignment_df[SOURCE_COLUMN].map(lambda p: Path(p).exists())
assignment_df["source_is_file"] = assignment_df[SOURCE_COLUMN].map(lambda p: Path(p).is_file())

missing_sources_df = assignment_df[
    ~assignment_df["source_exists"] | ~assignment_df["source_is_file"]
].copy()

if not missing_sources_df.empty:
    missing_sources_path = TABLES_DIR / "missing_sources_before_materialization.csv"
    missing_sources_df.to_csv(missing_sources_path, index=False)
    raise FileNotFoundError(
        f"Some source files are missing or are not regular files. "
        f"See: {missing_sources_path}"
    )

assignment_df = assignment_df.sort_values(
    ["proposed_folder", "cluster_label", "embedding_row"],
    ascending=[True, True, True],
).reset_index(drop=True)

prepared_assignment_path = TABLES_DIR / "prepared_folder_assignment_manifest.csv"
assignment_df.to_csv(prepared_assignment_path, index=False)

print(f"Prepared assignment rows: {len(assignment_df)}")
print(f"Unique proposed folders: {assignment_df['proposed_folder'].nunique()}")

Prepared assignment rows: 15705
Unique proposed folders: 356


# Build destination plan

In [8]:
destination_rows = []
used_destination_paths = set()

for _, row in assignment_df.iterrows():
    proposed_folder = sanitize_folder_name(row["proposed_folder"], fallback="_needs_review")
    source_path = Path(row[SOURCE_COLUMN])

    destination_folder = FINAL_OUTPUT_DIR / proposed_folder
    destination_filename = make_destination_filename(row, source_path)
    initial_destination_path = destination_folder / destination_filename
    destination_path = ensure_unique_destination_path(initial_destination_path, used_destination_paths)

    destination_rows.append({
        "embedding_row": int(row["embedding_row"]),
        "source_path": source_path.as_posix(),
        "destination_folder": destination_folder.as_posix(),
        "destination_path": destination_path.as_posix(),
        "destination_filename": destination_path.name,
        "proposed_folder": proposed_folder,
    })

destination_plan_df = pd.DataFrame(destination_rows)

materialization_plan_df = assignment_df.merge(
    destination_plan_df,
    on=["embedding_row", "source_path", "proposed_folder"],
    how="left",
    validate="one_to_one",
)

if materialization_plan_df["destination_path"].isna().any():
    raise ValueError("Some assignment rows did not receive a destination path.")

if materialization_plan_df["destination_path"].duplicated().any():
    duplicated_destinations = materialization_plan_df.loc[
        materialization_plan_df["destination_path"].duplicated(keep=False),
        "destination_path"
    ].head(10).tolist()

    raise ValueError(f"Duplicated destination paths were generated: {duplicated_destinations}")

materialization_plan_path = TABLES_DIR / "materialization_plan.csv"
materialization_plan_df.to_csv(materialization_plan_path, index=False)

print(f"Materialization plan rows: {len(materialization_plan_df)}")
print(f"Unique destination paths: {materialization_plan_df['destination_path'].nunique()}")
print(f"Output folders planned: {materialization_plan_df['proposed_folder'].nunique()}")

Materialization plan rows: 15705
Unique destination paths: 15705
Output folders planned: 356


# Materialize files

In [9]:
materialization_rows = []

for _, row in tqdm(
    materialization_plan_df.iterrows(),
    total=len(materialization_plan_df),
    desc="Materializing organized files",
):
    source_path = Path(row["source_path"])
    destination_path = Path(row["destination_path"])

    result = materialize_one_file(
        source_path=source_path,
        destination_path=destination_path,
        mode=MATERIALIZATION_MODE,
        on_existing=ON_EXISTING,
        allow_hardlink_fallback_to_copy=ALLOW_HARDLINK_FALLBACK_TO_COPY,
    )

    source_stat = source_path.stat() if source_path.exists() and source_path.is_file() else None
    destination_exists = destination_path.exists() or destination_path.is_symlink()

    if destination_exists and destination_path.is_file():
        destination_size = destination_path.stat().st_size
    elif destination_exists and destination_path.is_symlink():
        try:
            destination_size = destination_path.stat().st_size
        except Exception:
            destination_size = None
    else:
        destination_size = None

    materialization_rows.append({
        "embedding_row": int(row["embedding_row"]),
        "source_path": source_path.as_posix(),
        "destination_path": destination_path.as_posix(),
        "proposed_folder": row["proposed_folder"],
        "cluster_label": int(row["cluster_label"]),
        "assignment_category": row["assignment_category"],
        "status": result["status"],
        "operation_used": result["operation_used"],
        "error_type": result["error_type"],
        "error_message": result["error_message"],
        "source_size_bytes": int(source_stat.st_size) if source_stat is not None else None,
        "destination_exists": bool(destination_exists),
        "destination_size_bytes": int(destination_size) if destination_size is not None else None,
        "processed_at": now_iso(),
    })

materialization_result_df = pd.DataFrame(materialization_rows)

materialization_result_path = TABLES_DIR / "materialization_result.csv"
materialization_result_df.to_csv(materialization_result_path, index=False)

status_counts_df = (
    materialization_result_df
    .groupby(["status", "operation_used"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

status_counts_path = TABLES_DIR / "materialization_status_counts.csv"
status_counts_df.to_csv(status_counts_path, index=False)

status_counts_df

Materializing organized files:   0%|          | 0/15705 [00:00<?, ?it/s]

,status,operation_used,count
0,ok,hardlink,15705


# Validate materialized files

In [10]:
validation_rows = []

for _, row in tqdm(
    materialization_result_df.iterrows(),
    total=len(materialization_result_df),
    desc="Validating materialized files",
):
    source_path = Path(row["source_path"])
    destination_path = Path(row["destination_path"])

    source_exists = source_path.exists()
    destination_exists = destination_path.exists() or destination_path.is_symlink()

    source_size = source_path.stat().st_size if source_exists and source_path.is_file() else None

    try:
        destination_size = destination_path.stat().st_size if destination_exists else None
    except Exception:
        destination_size = None

    same_size = (
        source_size is not None and
        destination_size is not None and
        int(source_size) == int(destination_size)
    )

    validation_rows.append({
        "embedding_row": int(row["embedding_row"]),
        "source_path": source_path.as_posix(),
        "destination_path": destination_path.as_posix(),
        "source_exists": bool(source_exists),
        "destination_exists": bool(destination_exists),
        "source_size_bytes": int(source_size) if source_size is not None else None,
        "destination_size_bytes": int(destination_size) if destination_size is not None else None,
        "same_size": bool(same_size),
        "valid_materialization": bool(source_exists and destination_exists and same_size),
    })

materialization_validation_df = pd.DataFrame(validation_rows)

materialization_validation_path = TABLES_DIR / "materialization_validation.csv"
materialization_validation_df.to_csv(materialization_validation_path, index=False)

invalid_materialization_df = materialization_validation_df[
    ~materialization_validation_df["valid_materialization"]
].copy()

invalid_materialization_path = TABLES_DIR / "invalid_materialization.csv"
invalid_materialization_df.to_csv(invalid_materialization_path, index=False)

print(f"Validated files: {len(materialization_validation_df)}")
print(f"Invalid materializations: {len(invalid_materialization_df)}")

if not invalid_materialization_df.empty:
    display(invalid_materialization_df.head(30))

Validating materialized files:   0%|          | 0/15705 [00:00<?, ?it/s]

Validated files: 15705
Invalid materializations: 0


# Create per-folder manifests inside organized output

In [11]:
folder_manifest_rows = []

if CREATE_FOLDER_MANIFEST_FILES:
    for folder_name, group in tqdm(
        materialization_plan_df.groupby("proposed_folder", dropna=False),
        desc="Writing per-folder manifests",
    ):
        folder_path = FINAL_OUTPUT_DIR / folder_name
        folder_path.mkdir(parents=True, exist_ok=True)

        folder_group = group.copy()
        folder_result = materialization_result_df[
            materialization_result_df["embedding_row"].isin(folder_group["embedding_row"])
        ].copy()

        folder_manifest = folder_group.merge(
            folder_result[[
                "embedding_row",
                "destination_path",
                "status",
                "operation_used",
                "error_type",
                "error_message",
                "destination_exists",
                "destination_size_bytes",
            ]],
            on=["embedding_row", "destination_path"],
            how="left",
            validate="one_to_one",
        )

        folder_manifest_path = folder_path / "_folder_manifest.csv"
        folder_manifest.to_csv(folder_manifest_path, index=False)

        folder_manifest_rows.append({
            "proposed_folder": folder_name,
            "folder_path": folder_path.as_posix(),
            "folder_manifest_path": folder_manifest_path.as_posix(),
            "image_count": int(len(folder_manifest)),
        })

folder_manifest_index_df = pd.DataFrame(folder_manifest_rows)

folder_manifest_index_path = TABLES_DIR / "folder_manifest_index.csv"
folder_manifest_index_df.to_csv(folder_manifest_index_path, index=False)

folder_manifest_index_df.head(20)

Writing per-folder manifests:   0%|          | 0/356 [00:00<?, ?it/s]

,proposed_folder,folder_path,folder_manifest_path,image_count
0,_needs_review_noise,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,9138
1,cluster_00000_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,4
2,cluster_00000_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,2
3,cluster_00001_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,3
4,cluster_00001_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,2
5,cluster_00002_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,13
6,cluster_00002_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,1
7,cluster_00003_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,8
8,cluster_00003_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,1
9,cluster_00004_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,9


# Create per-folder metadata files


In [13]:
folder_metadata_rows = []

if CREATE_CLUSTER_METADATA_FILES:
    cluster_summary_lookup = {}

    if not cluster_summary_df.empty and "cluster_folder_stub" in cluster_summary_df.columns:
        for _, row in cluster_summary_df.iterrows():
            cluster_summary_lookup[str(row["cluster_folder_stub"])] = row.replace({np.nan: None}).to_dict()

    for folder_name, group in tqdm(
        materialization_plan_df.groupby("proposed_folder", dropna=False),
        desc="Writing per-folder metadata",
    ):
        folder_path = FINAL_OUTPUT_DIR / folder_name
        folder_path.mkdir(parents=True, exist_ok=True)

        unique_cluster_labels = sorted(group["cluster_label"].dropna().astype(int).unique().tolist())

        metadata = {
            "folder_name": folder_name,
            "folder_path": folder_path.as_posix(),
            "created_at": now_iso(),
            "materialization_mode": MATERIALIZATION_MODE,
            "source_column": SOURCE_COLUMN,
            "image_count": int(len(group)),
            "assignment_categories": (
                group.groupby("assignment_category", dropna=False)
                .size()
                .reset_index(name="count")
                .replace({np.nan: None})
                .to_dict(orient="records")
            ),
            "cluster_labels": unique_cluster_labels,
            "requires_manual_review_count": int(group["requires_manual_review"].map(safe_bool).sum()),
            "cluster_summary": cluster_summary_lookup.get(folder_name),
            "sample_items": dataframe_to_json_records(
                group[[
                    "embedding_row",
                    "relative_path",
                    "source_path",
                    "destination_path",
                    "cluster_label",
                    "cluster_probability",
                    "outlier_score",
                    "requires_manual_review",
                    "assignment_category",
                ]],
                max_rows=25,
            ),
        }

        metadata_path = folder_path / "_cluster_info.json"

        with metadata_path.open("w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        folder_metadata_rows.append({
            "proposed_folder": folder_name,
            "folder_path": folder_path.as_posix(),
            "metadata_path": metadata_path.as_posix(),
            "image_count": int(len(group)),
        })

folder_metadata_index_df = pd.DataFrame(folder_metadata_rows)

folder_metadata_index_path = TABLES_DIR / "folder_metadata_index.csv"
folder_metadata_index_df.to_csv(folder_metadata_index_path, index=False)

folder_metadata_index_df.head(20)


Writing per-folder metadata:   0%|          | 0/356 [00:00<?, ?it/s]

,proposed_folder,folder_path,metadata_path,image_count
0,_needs_review_noise,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,9138
1,cluster_00000_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,4
2,cluster_00000_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,2
3,cluster_00001_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,3
4,cluster_00001_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,2
5,cluster_00002_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,13
6,cluster_00002_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,1
7,cluster_00003_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,8
8,cluster_00003_unknown_review,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,1
9,cluster_00004_unknown,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,9


# Create output contact sheets by folder


In [14]:
folder_contact_sheet_rows = []

if CREATE_OUTPUT_CONTACT_SHEET_INDEX:
    contact_sheet_output_dir = FINAL_OUTPUT_DIR / "_contact_sheets"
    contact_sheet_output_dir.mkdir(parents=True, exist_ok=True)

    folder_sizes_df = (
        materialization_plan_df
        .groupby("proposed_folder", dropna=False)
        .size()
        .reset_index(name="image_count")
        .sort_values("image_count", ascending=False)
        .reset_index(drop=True)
    )

    selected_folders = folder_sizes_df.head(MAX_FOLDER_CONTACT_SHEETS)["proposed_folder"].tolist()

    for folder_name in tqdm(selected_folders, desc="Creating output folder contact sheets"):
        group = materialization_plan_df[
            materialization_plan_df["proposed_folder"].eq(folder_name)
        ].copy()

        group = group.sort_values(
            ["cluster_probability", "outlier_score"],
            ascending=[False, True],
        ).head(CONTACT_SHEET_MAX_IMAGES_PER_FOLDER)

        destination_paths = group["destination_path"].dropna().tolist()

        output_path = contact_sheet_output_dir / f"{sanitize_filename_component(folder_name, fallback='folder')}.jpg"

        created = create_contact_sheet(
            image_paths=destination_paths,
            output_path=output_path,
            thumb_size=CONTACT_SHEET_THUMB_SIZE,
            columns=CONTACT_SHEET_COLUMNS,
            title=f"{folder_name} | n={len(materialization_plan_df[materialization_plan_df['proposed_folder'].eq(folder_name)])}",
        )

        folder_contact_sheet_rows.append({
            "proposed_folder": folder_name,
            "image_count": int(len(materialization_plan_df[materialization_plan_df["proposed_folder"].eq(folder_name)])),
            "sampled_image_count": int(len(destination_paths)),
            "created": bool(created),
            "contact_sheet_path": output_path.as_posix() if created else None,
        })

folder_contact_sheets_df = pd.DataFrame(folder_contact_sheet_rows)

folder_contact_sheets_path = TABLES_DIR / "output_folder_contact_sheets.csv"
folder_contact_sheets_df.to_csv(folder_contact_sheets_path, index=False)

folder_contact_sheets_df.head(20)

Creating output folder contact sheets:   0%|          | 0/300 [00:00<?, ?it/s]

,proposed_folder,image_count,sampled_image_count,created,contact_sheet_path
0,_needs_review_noise,9138,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
1,cluster_00183_unknown,368,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
2,cluster_00171_unknown,268,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
3,cluster_00169_unknown,264,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
4,cluster_00181_unknown,192,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
5,cluster_00172_unknown,182,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
6,cluster_00110_unknown,170,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
7,cluster_00183_unknown_review,137,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
8,cluster_00170_unknown,118,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...
9,cluster_00179_unknown,101,24,True,/home/ssparzival/Proyectos/GitHub/Automatic-Or...


# Optionally copy clustering contact sheets into output

In [15]:
copied_contact_sheet_rows = []

if COPY_CLUSTER_CONTACT_SHEETS_TO_OUTPUT:
    review_sheet_dir = FINAL_OUTPUT_DIR / "_clustering_review_sheets"
    review_sheet_dir.mkdir(parents=True, exist_ok=True)

    contact_sheet_sources = []

    if not cluster_contact_sheets_df.empty and "contact_sheet_path" in cluster_contact_sheets_df.columns:
        for _, row in cluster_contact_sheets_df.iterrows():
            path_value = row.get("contact_sheet_path")
            if pd.notna(path_value) and Path(path_value).exists():
                contact_sheet_sources.append(Path(path_value))

    if not special_contact_sheets_df.empty and "contact_sheet_path" in special_contact_sheets_df.columns:
        for _, row in special_contact_sheets_df.iterrows():
            path_value = row.get("contact_sheet_path")
            if pd.notna(path_value) and Path(path_value).exists():
                contact_sheet_sources.append(Path(path_value))

    seen_sheet_names = set()

    for source_path in tqdm(contact_sheet_sources, desc="Copying clustering contact sheets"):
        safe_name = sanitize_filename_component(source_path.name, fallback="contact_sheet")
        destination_path = review_sheet_dir / safe_name

        if destination_path.name in seen_sheet_names or destination_path.exists():
            destination_path = review_sheet_dir / f"{source_path.stem}__{stable_short_hash(source_path.as_posix(), 8)}{source_path.suffix}"

        seen_sheet_names.add(destination_path.name)

        try:
            shutil.copy2(source_path, destination_path)
            status = "ok"
            error_type = None
            error_message = None
        except Exception as exc:
            status = "error"
            error_type = type(exc).__name__
            error_message = str(exc)

        copied_contact_sheet_rows.append({
            "source_contact_sheet_path": source_path.as_posix(),
            "destination_contact_sheet_path": destination_path.as_posix(),
            "status": status,
            "error_type": error_type,
            "error_message": error_message,
        })

copied_contact_sheets_df = pd.DataFrame(copied_contact_sheet_rows)

copied_contact_sheets_path = TABLES_DIR / "copied_clustering_contact_sheets.csv"
copied_contact_sheets_df.to_csv(copied_contact_sheets_path, index=False)

copied_contact_sheets_df.head(20)

Copying clustering contact sheets:   0%|          | 0/190 [00:00<?, ?it/s]

,source_contact_sheet_path,destination_contact_sheet_path,status,error_type,error_message
0,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
1,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
2,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
3,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
4,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
5,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
6,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
7,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
8,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None
9,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,/home/ssparzival/Proyectos/GitHub/Automatic-Or...,ok,None,None


# Create organized output README


In [18]:
if CREATE_OUTPUT_README:
    folder_counts_df = (
        materialization_plan_df
        .groupby(["assignment_category", "proposed_folder"], dropna=False)
        .size()
        .reset_index(name="image_count")
        .sort_values(["assignment_category", "image_count"], ascending=[True, False])
        .reset_index(drop=True)
    )

    operation_counts_df = (
        materialization_result_df
        .groupby(["status", "operation_used"], dropna=False)
        .size()
        .reset_index(name="count")
        .sort_values("count", ascending=False)
        .reset_index(drop=True)
    )

    readme_lines = [
        "# Organized Anime Character Image Output",
        "",
        "This directory was generated by the automatic anime character organization pipeline.",
        "",
        "## Important",
        "",
        "- The original image directory was not modified.",
        f"- Materialization mode used: `{MATERIALIZATION_MODE}`.",
        f"- Source column used: `{SOURCE_COLUMN}`.",
        f"- Generated at: `{now_iso()}`.",
        f"- Previous clustering run: `{PREVIOUS_RUN_DIR.as_posix()}`.",
        f"- Materialization run: `{RUN_DIR.as_posix()}`.",
        "",
        "## Output structure",
        "",
        "- Character-like clusters are stored as `cluster_XXXXX_unknown/` folders.",
        "- Ambiguous noise images are stored under `_needs_review_noise/`.",
        "- Cluster members requiring manual review are stored under folders ending in `_review`.",
        "- Each folder may contain `_folder_manifest.csv` and `_cluster_info.json`.",
        "- `_contact_sheets/` contains visual summaries of the generated folders.",
        "- `_clustering_review_sheets/` contains contact sheets copied from the clustering notebook, if enabled.",
        "",
        "## Operation counts",
        "",
        operation_counts_df.to_markdown(index=False),
        "",
        "## Folder counts preview",
        "",
        folder_counts_df.head(80).to_markdown(index=False),
        "",
    ]

    readme_path = FINAL_OUTPUT_DIR / "README.md"

    with readme_path.open("w", encoding="utf-8") as f:
        f.write("\n".join(readme_lines))

    print(f"README written to: {readme_path}")
else:
    readme_path = None

README written to: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/organized_output/anime_organized_20260523_220036/README.md


# Save global indexes and reports


In [19]:
folder_counts_df = (
    materialization_plan_df
    .groupby(["assignment_category", "proposed_folder"], dropna=False)
    .size()
    .reset_index(name="image_count")
    .sort_values(["assignment_category", "image_count"], ascending=[True, False])
    .reset_index(drop=True)
)

folder_counts_path = TABLES_DIR / "final_folder_counts.csv"
folder_counts_df.to_csv(folder_counts_path, index=False)

operation_counts_df = (
    materialization_result_df
    .groupby(["status", "operation_used"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)

operation_counts_path = TABLES_DIR / "operation_counts.csv"
operation_counts_df.to_csv(operation_counts_path, index=False)

global_index_path = FINAL_OUTPUT_DIR / "_global_materialization_index.csv"

global_index_columns = [
    "embedding_row",
    "relative_path",
    "source_path",
    "destination_path",
    "proposed_folder",
    "cluster_label",
    "cluster_probability",
    "outlier_score",
    "requires_manual_review",
    "assignment_category",
]

materialization_plan_df[global_index_columns].to_csv(global_index_path, index=False)

print(f"Final folder counts: {folder_counts_path}")
print(f"Operation counts: {operation_counts_path}")
print(f"Global output index: {global_index_path}")

Final folder counts: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materialization_20260523_220036/tables/final_folder_counts.csv
Operation counts: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materialization_20260523_220036/tables/operation_counts.csv
Global output index: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/organized_output/anime_organized_20260523_220036/_global_materialization_index.csv


# Final consistency checks


In [20]:
expected_rows = len(materialization_plan_df)

successful_or_skipped_df = materialization_result_df[
    materialization_result_df["status"].isin(["ok", "skipped_existing"])
].copy()

operation_ok_count = int(materialization_result_df["status"].eq("ok").sum())
skipped_count = int(materialization_result_df["status"].eq("skipped_existing").sum())
error_count = int(materialization_result_df["status"].eq("error").sum())

valid_materialization_count = int(materialization_validation_df["valid_materialization"].sum())
invalid_materialization_count = int((~materialization_validation_df["valid_materialization"]).sum())

destination_paths_unique = materialization_plan_df["destination_path"].nunique() == len(materialization_plan_df)

checks = {
    "final_output_dir_exists": FINAL_OUTPUT_DIR.exists(),
    "materialization_plan_rows_match_assignment": bool(len(materialization_plan_df) == len(assignment_df)),
    "materialization_result_rows_match_plan": bool(len(materialization_result_df) == len(materialization_plan_df)),
    "materialization_validation_rows_match_plan": bool(len(materialization_validation_df) == len(materialization_plan_df)),
    "destination_paths_unique": bool(destination_paths_unique),
    "no_materialization_errors": bool(error_count == 0),
    "all_destination_files_valid": bool(invalid_materialization_count == 0),
    "global_index_exists": global_index_path.exists(),
    "folder_counts_exists": folder_counts_path.exists(),
    "operation_counts_exists": operation_counts_path.exists(),
}

if CREATE_OUTPUT_README:
    checks["readme_exists"] = readme_path.exists()

if CREATE_FOLDER_MANIFEST_FILES:
    checks["folder_manifest_index_exists"] = folder_manifest_index_path.exists()

if CREATE_CLUSTER_METADATA_FILES:
    checks["folder_metadata_index_exists"] = folder_metadata_index_path.exists()

if CREATE_OUTPUT_CONTACT_SHEET_INDEX:
    checks["output_folder_contact_sheets_exists"] = folder_contact_sheets_path.exists()

failed_checks = {key: value for key, value in checks.items() if not value}

checks_path = REPORTS_DIR / "consistency_checks.json"

with checks_path.open("w", encoding="utf-8") as f:
    json.dump(checks, f, ensure_ascii=False, indent=2)

if failed_checks:
    raise RuntimeError(f"Consistency checks failed: {failed_checks}")

checks

{'final_output_dir_exists': True,
 'materialization_plan_rows_match_assignment': True,
 'materialization_result_rows_match_plan': True,
 'materialization_validation_rows_match_plan': True,
 'destination_paths_unique': True,
 'no_materialization_errors': True,
 'all_destination_files_valid': True,
 'global_index_exists': True,
 'folder_counts_exists': True,
 'operation_counts_exists': True,
 'readme_exists': True,
 'folder_manifest_index_exists': True,
 'folder_metadata_index_exists': True,
 'output_folder_contact_sheets_exists': True}

# Save run summary

In [21]:
summary = {
    "run_id": RUN_ID,
    "created_at": now_iso(),
    "project_dir": PROJECT_DIR.as_posix(),
    "previous_run_dir": PREVIOUS_RUN_DIR.as_posix(),
    "run_dir": RUN_DIR.as_posix(),
    "final_output_dir": FINAL_OUTPUT_DIR.as_posix(),
    "materialization_mode": MATERIALIZATION_MODE,
    "allow_hardlink_fallback_to_copy": bool(ALLOW_HARDLINK_FALLBACK_TO_COPY),
    "on_existing": ON_EXISTING,
    "source_column": SOURCE_COLUMN,
    "input_assignment_rows": int(len(folder_assignment_df)),
    "materialization_plan_rows": int(len(materialization_plan_df)),
    "operation_ok_count": int(operation_ok_count),
    "skipped_existing_count": int(skipped_count),
    "error_count": int(error_count),
    "valid_materialization_count": int(valid_materialization_count),
    "invalid_materialization_count": int(invalid_materialization_count),
    "unique_output_folders": int(materialization_plan_df["proposed_folder"].nunique()),
    "python": sys.version,
    "platform": platform.platform(),
    "outputs": {
        "final_output_dir": FINAL_OUTPUT_DIR.as_posix(),
        "prepared_assignment": prepared_assignment_path.as_posix(),
        "materialization_plan": materialization_plan_path.as_posix(),
        "materialization_result": materialization_result_path.as_posix(),
        "materialization_status_counts": status_counts_path.as_posix(),
        "materialization_validation": materialization_validation_path.as_posix(),
        "invalid_materialization": invalid_materialization_path.as_posix(),
        "folder_counts": folder_counts_path.as_posix(),
        "operation_counts": operation_counts_path.as_posix(),
        "global_output_index": global_index_path.as_posix(),
        "consistency_checks": checks_path.as_posix(),
    },
}

if CREATE_FOLDER_MANIFEST_FILES:
    summary["outputs"]["folder_manifest_index"] = folder_manifest_index_path.as_posix()

if CREATE_CLUSTER_METADATA_FILES:
    summary["outputs"]["folder_metadata_index"] = folder_metadata_index_path.as_posix()

if CREATE_OUTPUT_CONTACT_SHEET_INDEX:
    summary["outputs"]["output_folder_contact_sheets"] = folder_contact_sheets_path.as_posix()

if COPY_CLUSTER_CONTACT_SHEETS_TO_OUTPUT:
    summary["outputs"]["copied_clustering_contact_sheets"] = copied_contact_sheets_path.as_posix()

if CREATE_OUTPUT_README:
    summary["outputs"]["readme"] = readme_path.as_posix()

summary_path = REPORTS_DIR / "summary.json"

with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved outputs:")
for key, value in summary["outputs"].items():
    print(f"{key}: {value}")

print(f"summary: {summary_path.as_posix()}")

Saved outputs:
final_output_dir: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/organized_output/anime_organized_20260523_220036
prepared_assignment: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materialization_20260523_220036/tables/prepared_folder_assignment_manifest.csv
materialization_plan: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materialization_20260523_220036/tables/materialization_plan.csv
materialization_result: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materialization_20260523_220036/tables/materialization_result.csv
materialization_status_counts: /home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/runs/05_folder_materializa

# Inspect final operation counts


In [22]:
operation_counts_df

,status,operation_used,count
0,ok,hardlink,15705


# Inspect final folder counts


In [23]:
folder_counts_df.head(80)

,assignment_category,proposed_folder,image_count
0,cluster,cluster_00183_unknown,368
1,cluster,cluster_00171_unknown,268
2,cluster,cluster_00169_unknown,264
3,cluster,cluster_00181_unknown,192
4,cluster,cluster_00172_unknown,182
...,...,...,...
75,cluster,cluster_00091_unknown,15
76,cluster,cluster_00143_unknown,15
77,cluster,cluster_00164_unknown,15
78,cluster,cluster_00184_unknown,15


# Inspect any errors

In [24]:
errors_df = materialization_result_df[
    materialization_result_df["status"].eq("error")
].copy()

if errors_df.empty:
    print("No materialization errors detected.")
else:
    errors_df[[
        "embedding_row",
        "source_path",
        "destination_path",
        "proposed_folder",
        "operation_used",
        "error_type",
    ]]

No materialization errors detected.


# Final output location


In [25]:
print("Final organized output:")
print(FINAL_OUTPUT_DIR.as_posix())

print("\nGlobal index:")
print(global_index_path.as_posix())

if CREATE_OUTPUT_README:
    print("\nREADME:")


Final organized output:
/home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/organized_output/anime_organized_20260523_220036

Global index:
/home/ssparzival/Proyectos/GitHub/Automatic-Organization-of-Anime-Character-Images/anime_character_pipeline/organized_output/anime_organized_20260523_220036/_global_materialization_index.csv

README:
